<a href="https://colab.research.google.com/github/angelgreen/paperAndNote/blob/master/asr_paper_%E5%A4%A7%E7%BA%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 面向移动端离线中文短语音识别的混合建模与解码策略研究


## Introduction
随着移动设备上语音交互应用的不断发展，离线自动语音识别（Automatic Speech Recognition, ASR）在语音输入、语音命令和人机交互等场景中具有重要的应用价值。与依赖云端服务的识别系统相比，端侧离线 ASR 可以在不上传用户语音的情况下完成识别，并减少网络连接带来的延迟。然而，移动设备的计算能力、内存和能耗预算相对有限，因此，如何在保证识别准确率的同时控制模型的推理开销，是端侧 ASR 系统设计中需要考虑的重要问题。对于短语音识别任务，解码过程的额外开销同样值得关注，因为系统不仅需要准确地识别语音内容，还需要在有限的时间内返回识别结果。
近年来，基于卷积神经网络的端到端 ASR 模型取得了显著进展。Jasper、QuartzNet、ContextNet 和 Citrinet 等模型探索了基于卷积声学编码器的语音特征建模方法 [1–4]。其中，Citrinet 采用一维时间通道可分离卷积、残差连接和 Squeeze-and-Excitation 模块，并结合连接时序分类（Connectionist Temporal Classification, CTC）目标进行训练，在多个语音识别基准上取得了有竞争力的结果 [4]。
CTC 为输入语音帧与输出 Token 序列之间的对齐提供了有效的建模方式，并允许通过贪心解码等方法进行高效推理。然而，CTC 输出层在给定输入后对各帧输出作条件独立性假设，单纯依赖 CTC 贪心解码并不能显式利用已经生成的完整 Token 历史来建模输出序列之间的依赖关系。
另一方面，基于 Transformer 的 Encoder–Decoder 架构已被广泛应用于端到端语音识别。以 Whisper 为例，该模型利用 Transformer 编码器提取音频特征，并通过自回归解码器生成文本，在大规模弱监督训练条件下表现出较强的识别能力 [5]。在这类架构中，解码器通过交叉注意力机制利用编码器输出的声学信息，并结合此前生成的 Token 预测下一个 Token。这种自回归建模方式能够显式利用输出序列的历史信息，但生成过程需要逐步预测 Token，因而会引入额外的串行解码开销。由此可见，CTC 解码与 Transformer 自回归解码在序列建模能力和推理方式上具有不同的特点，如何在识别准确率与推理开销之间进行权衡，是值得进一步研究的问题。
基于上述考虑，本文面向移动端离线中文短语音识别，构建了一种结合 Citrinet 编码器与 Transformer 解码器的端到端识别系统，并研究不同解码策略对识别效果和端侧推理开销的影响。在共享模型参数的条件下，本文对三种解码策略进行比较：CTC 贪心解码、Transformer 自回归解码，以及基于 CTC N-best 候选生成和 Transformer 重排序的两阶段解码方法。通过在中文语音测试集上评估字符错误率（Character Error Rate, CER），并测量移动端实际推理延迟，本文分析不同解码策略在识别准确率与计算开销方面的权衡。
本文的主要工作包括：
1. 构建基于 Citrinet 编码器与 Transformer 解码器的端到端中文语音识别系统，并将其部署到移动端离线推理环境中。
2. 在统一的模型与评估设置下，对 CTC 贪心解码、Transformer 自回归解码和 CTC N-best 与 Transformer 重排序三种策略进行系统比较。

# Related Work and Background

## Convolutional Acoustic Models for ASR
卷积神经网络（Convolutional Neural Networks, CNNs）已广泛应用于端到端自动语音识别（Automatic Speech Recognition, ASR）系统中的声学建模。对于 Log-Mel 滤波器组等声学特征，卷积操作能够提取相邻时序帧之间的局部特征。通过堆叠多个卷积层并合理设计卷积核，网络可以逐步扩大时序感受野，从而捕获不同时间尺度上的声学模式。此外，卷积运算支持对不同时间位置进行并行计算，因此，基于 CNN 的声学编码器在特征提取效率和模型结构设计方面具有一定优势。
Jasper [1] 提出了一种深层卷积声学模型，其主要由一维卷积层、BatchNorm、ReLU 激活函数、Dropout 和残差连接组成。在此基础上，QuartzNet [2] 采用一维时间通道可分离卷积（1D Time-Channel Separable Convolution）替代标准卷积。该操作将卷积分解为沿时间维度进行的深度卷积（Depthwise Convolution）和用于通道间信息融合的逐点卷积（Pointwise Convolution），从而减少模型参数量，并保留对时序声学特征的提取能力。随后，ContextNet [3] 在卷积声学编码器中引入 Squeeze-and-Excitation（SE）模块，通过对通道特征进行自适应重标定，进一步利用全局上下文信息。这些研究表明，通过合理设计卷积结构，可以在声学特征建模能力与模型复杂度之间进行权衡。
Citrinet [4] 进一步发展了基于 QuartzNet 的卷积架构，将一维时间通道可分离卷积、残差连接和 SE 模块结合起来，并采用子词（Subword）建模方式和连接时序分类（Connectionist Temporal Classification, CTC）目标实现端到端语音识别。Citrinet 在 LibriSpeech、TED-LIUM2 和 AISHELL-1 等多个语音识别基准上进行了评估，并取得了具有竞争力的识别性能。上述结构特点使 Citrinet 成为值得研究的声学编码器，尤其适用于需要同时关注识别准确率与计算资源开销的应用场景。
本文采用 Citrinet 作为声学编码器，并进一步研究其与 Transformer 解码器的结合方式。在此基础上，本文比较不同解码策略在离线中文短语音识别任务中的表现，重点分析识别准确率与推理开销之间的权衡，为移动端离线语音识别系统的设计与部署提供实验依据。

## CTC-Based Speech Recognition
连接时序分类（Connectionist Temporal Classification, CTC）是一种用于处理未对齐序列数据的训练目标，由 Graves 等人提出 [5]。在端到端语音识别中，声学编码器将输入语音映射为长度为 (T) 的特征序列，并在每个时间步预测字符或子词 Token 的概率分布。由于输入帧与目标文本之间通常不存在显式的逐帧对齐关系，CTC 引入空白符（Blank）并定义路径折叠操作，将不同的帧级预测路径映射为同一目标序列。通过对所有能够映射到目标文本的有效路径求和，CTC 可以在不需要人工提供帧级对齐标注的情况下，直接优化目标文本的条件概率：
$$
  P(y\mid x) = \sum_{\pi\in\mathcal{B}^{-1}(y)}
          P(\pi\mid x),
          \qquad
          \mathcal{L}_{\mathrm{CTC}}
          =-\log P(y\mid x),
$$
其中，$(x)$表示输入语音特征，$(y)$表示目标Token序列，$(\pi)$表示包含Blank的帧级预测路径，$(\mathcal{B})$表示将重复标签合并并移除Blank的折叠操作，$(\mathcal{B}^{-1}(y))$表示所有能够映射为$(y)$的有效路径。
在推理阶段，CTC 可以采用贪心解码（Greedy Decoding），即在每个时间步独立选择概率最大的标签，再通过路径折叠得到最终文本。该方法实现简单，计算开销较低，适合对推理效率有要求的应用场景。然而，贪心解码并不保证获得概率最大的输出文本。这是因为 CTC 对目标序列的概率由所有对应对齐路径的概率共同决定，而贪心解码仅选择逐帧最优标签构成的单条路径。因此，概率最高的单条路径经过折叠后，不一定对应概率最高的目标序列。
尽管 CTC 能够有效处理输入与输出之间的对齐问题，但其帧级输出在给定输入序列后采用条件独立性假设，难以显式建模目标 Token 之间的历史依赖关系。为此，本文进一步引入 Transformer Decoder，研究自回归解码以及基于 CTC 候选序列的 Transformer 重排序方法，并在后续实验中比较不同解码策略对识别准确率和移动端推理开销的影响。
## Attention-Based Decoding and N-best Rescoring
基于注意力机制的 Encoder–Decoder 架构是端到端语音识别的重要技术路线。给定输入声学特征$(x)$，编码器首先提取声学表示$H=\operatorname{Encoder}(x)$，解码器则结合编码器输出和已生成的Token序列，逐步预测下一个Token，其条件概率可表示为：
$$
  P(y\mid x) = \prod_{u=1}^{U} P(y_u \mid y_{<u},H)
$$
其中，$y_u$表示第$u$个输出Token，$y_{<u}$表示此前生成的Token序列。Transformer Decoder通过自注意机制建模输出序列内部的依赖关系，并通过交叉注意力机制利用编码器提取的声学信息。与CTC贪心解码相比，自回归解码能够显式利用输出历史进行序列预测，但需要逐步生成 Token，其推理开销会受到输出序列长度和解码策略的影响。

为结合 CTC 与注意力模型的不同特点，Hori 等人 [6] 提出了联合 CTC/Attention 解码方法，通过融合两种模型的评分进行候选搜索。与联合解码不同，WeNet [7] 采用两阶段解码策略：首先利用 CTC 生成 N-best 候选序列，再由 Attention Decoder 对候选进行重评分，并选择最终识别结果。对于给定候选序列 $y$，注意力解码器可以计算其条件对数概率：
$$
  S_{\mathrm{Att}}(y\mid x) =
   \sum_{u=1}^{U}
   \log P(y_u\mid y_{<u}, H)
$$
在此基础上，可以将注意力模型的序列得分与 CTC 得分进行融合，对候选序列重新排序。该过程不需要重新自回归生成候选文本，而是对已有候选进行评分，因此可以采用 Teacher Forcing 的方式并行计算候选序列的 Token 概率。不过，重排序的效果受到候选集合质量、评分方式和候选数量等因素的影响，同时也会引入额外的推理计算。

针对移动端离线语音识别，模型设计不仅需要考虑识别准确率，还需要关注解码延迟和设备资源开销。近年来，已有研究进一步探索了面向端侧语音识别的 Transformer 候选重排序方法 [8]。基于上述研究，本文在 Citrinet 编码器与 Transformer 解码器组成的识别系统中，比较 CTC 贪心解码、Transformer 自回归解码和 CTC N-best 与 Transformer 重排序三种策略，并通过识别准确率和移动端实测延迟分析不同解码方法的适用性。

# System Architecture and Decoding Strategies
## Overall Architecture

## Citrinet Encoder
本文采用 Citrinet 作为声学编码器，使用窗口25ms，频移10ms，将输入语音转换为 Log-Mel 80维度的特征序列，并通过多层一维时间通道可分离卷积提取声学表示。经过编码器的时间下采样后，得到隐藏状态序列 H，并分别送入 CTC 输出层与 Transformer Decoder。两个分支共享声学编码器，使模型能够在联合训练过程中学习适用于帧级分类和自回归序列预测的声学表示。

## Joint Training Objective
设输入的音频特征为$X$，编码器输出为：
$$
  H = \operatorname{Encoder}(X)
$$
CTC Head根据$H$计算帧级输出概率：
$$
  P_{CTC}(\cdot | H)
$$
Transformer Decoder则根据声学表示和目标序列的历史Token，预测下一个Token：
$$
  P_{Tr}(y_u | y_{<u}, H)
$$
因此，可以联合训练目标写为：
$$
  \mathcal{L} = \lambda_{CTC} \mathcal{L}_{CTC} + \lambda_{Seq}\mathcal{L}_{Seq}
$$
其中，第一项让Encoder学习CTC所需的帧级对齐和分类能力，第二项通过Decoder的自回归目标学习Token序列之间的依赖关系。

CTC的损失函数可以写成：
$$
\mathcal{L}_{\mathrm{CTC}} = -\log\sum_{\pi\in\mathcal{B}^{-1}(y)}P(\pi\mid H)
$$
Seq2Seq的交叉熵损失函数可以写成：
$$
  \mathcal{L}_{\mathrm{Seq}} = -\sum_{u=1}^{U} \log P(y_u | y_{<u}, H)
$$
具体实现中，Decoder 在训练时使用 Teacher Forcing：输入移位后的目标 Token 序列，预测对应的下一个 Token。我们也使用一定的label Smooth，具体为0.02。
联合训练使声学编码器同时受到 CTC 目标与自回归序列预测目标的约束。CTC 分支提供基于帧级概率的建模与解码路径，Transformer Decoder 则学习基于历史 Token 的条件序列分布。本文采用该联合训练目标，为后续不同解码策略的比较提供统一的模型基础。
其中，当前训练中，$\lambda_{\mathrm{CTC}}$设置为0.4，$\lambda_{\mathrm{Seq}}$设置为0.6。

## Decoding Strategies

本文采用统一的 CTC-Transformer Encoder-Decoder 架构，通过不同的解码策略实现不同精度和计算开销之间的权衡。由于三种解码方式共享相同的 Citrinet Encoder 和 Transformer Decoder 参数，因此无需重新训练模型即可根据应用场景选择不同推理模式。

### CTC Greedy Decoding
给定Encoder 输出，先得到每个时间步的类别概率：
$$
  p_t(k) = P(\pi_k = k | X)
$$
逐帧选取最大概率类别：
$$
  \hat{\pi}_t = \mathop{\arg\max}\limits_{k} p_t(k)
$$
然后应用CTC路径折叠操作，合并连续重复Token并移除Blank，得到最终的文本。Greedy只选择一条逐帧最优路径，并不保证获得概率最大的完整文本序列。

### CTC Prefix Beam Seach + Transformer Rescoring
为结合 CTC 解码与 Transformer Decoder 的序列建模能力，本文采用基于 CTC Prefix Beam Search 和 Transformer 重评分的两阶段解码策略。该方法首先利用 CTC 从输入音频中生成多个候选文本，再利用 Transformer Decoder 对候选序列进行评分，并根据评分结果确定最终识别文本。

第一阶段：CTC 候选生成。给定输入音频特征$X$，首先通过Citrinet编码器提取声学表示：
$$
  H = \operatorname{Encoder}(X)
$$
CTC输出层根据$H$计算各时间步的类别概率分布，并使用Prefix Beam Search搜索高概率输出序列，得到包括$N$个候选文本的集合：
$$
  \{ y^{(1)}, y^{(2)}, \ldots, y^{(N)} \}
$$
在该阶段，CTC主要用于生成具有竞争力的候选文本，降低直接搜索最终文本的难度。与CTC Greedy Decoding 仅保留一条逐帧最优路径不同。Prefix Beam Search 能够保留多个候选，从而为后续重排序提供更多选择。

第二阶段：Transformer候选重评分。获得候选集合后，本文将候选文本组织为一个批次，并将其与对应的声学表示$H$一同输入Transformer Decoder。由于候选序列已经给定，Decoder可以使用Teacher Forcing计算每个候选的条件对数概率，而不需要重新执行自回归文本生成。
$$
  S_{Tr}(y^{(i)}|X) = \sum_{u=1}^{U_i} \log P_{\theta}(y_u^{(i)}| y_{<u}^{(i)}, H)
$$
其中，$y_u^{(i)}$表示第$(i)$个候选在第$(u)$个位置的目标Token，$y_{<u}^{(i)}$ 表示该位置之前的Token序列，$(U_i)$表示候选的有效输出长度，$(\theta)$表示Transformer Decoder的参数。计算时仅对候选的有效目标位置进行评分，并根据实际实现处理起始Token、结束Token与填充位置。
在文本采用的解码策略中，CTC分支用于生成候选集合，最终候选选择依据Transformer Decoder的条件对数概率进行排序：
$$
  \hat{y} = \mathop{\arg\max}\limits_{y^{(i)} \in y_N} S_{Tr}(y^{(i)}|X)
$$

其中，在Transformer候选评分阶段，本文进一步利用批量Cross-Attention和广播机制优化多个候选的并行计算。
因此，该方法并非将CTC与Transformer的分数进行加权融合，而是利用CTC完成候选搜索，并利用Transformer对已有候选进行重新排序。

其中，在Transformer候选评分阶段，本文进一步利用批量Cross-Attention和广播机制优化多个候选的并行计算。

为降低候选重评分阶段的计算开销，本文采用批量交叉注意力计算策略。为提高候选重评分阶段的计算效率，本文将同一段音频对应的多个候选组织为一个批次，并复用该音频的声学编码效果。在此过程中，声学表示经过交叉注意力中的Key和Value投影，而各候选文本分别产生相应的Query。由于所有候选对应同一段音频，其交叉注意力所使用的声学Key和Value具有相同的来源，无需为每个候选重新执行声学编码。

具体而言，设候选批次大小为$N$，分组查询注意力中的分组为$G$，每组的Query Head数为$n_h$，声学序列长度为$T_x$，候选文本的输入长度为$T_y$，单个注意力头的维度为$d_h$。经过相应的线性投影和维度整理后，Query、Key和Value的形状分别表示为：
$$
  Q \in \mathbb{R}^{N \times G \times n_h \times T_y \times d_h}
$$
$$
  K,V \in \mathbb{R}^{1 \times G \times 1 \times T_x \times d_h}
$$
这里的第一维表示单个音频样本，由于 N 个候选共享同一声学表示，因此无需沿候选维复制。
其中，Query的批次维度对应不同候选，而Key和Value的批次维度为1，表示当前批次中的所有候选共享同一段音频的声学表示。交叉注意力首先计算Query与Key之间的相关性，再通过Softmax得到注意力权重，并对Value进行加权求和：
$$
  A = \operatorname{Softmax}
      \left(
        s QK^T
      \right),
      \qquad
      C = AV,
$$
在实现层面，本文利用Tensorflow的tf.matmul广播机制进行上述矩阵计算。当Query的候选批次维度为$N$，而Key和Value对应维度为1时，矩阵乘法可在计算过程中自动完成维度广播，无需显式复制$N$份声学Key和Value张量。这样既可以在一次批量调用中完成多个候选的交叉注意力计算，也避免了对相同声学Key和Value进行显式重复存储。

需要指出的式，广播机制减少的是显式张量复制需求，并不会消除各候选自身的注意力计算。每个候选仍需根据其Query计算相应的注意力分数与上下文表示。因此，批量重评分的实际效率仍取决于候选数量、候选长度、模型计算开销及具体推理后端。

通过上述解码过程，本文结合了 CTC 在候选搜索阶段的高效性与 Transformer Decoder 在序列建模方面的优势，并针对移动端离线 ASR 场景设计了候选批量重评分机制，在保证识别精度的同时降低多候选推理带来的计算开销。

### transformer Autoregressive Decoding with KV Cache
解码从 BOS 开始，每一步利用已生成的 Token 和声学表示 H，预测下一个 Token，直到生成 EOS 或达到最大输出长度。

如果每一步都重新计算整个历史 Token 序列的自注意力，那么前面 Token 对应的 Key 和 Value 会被反复计算。KV Cache 则保存各层先前 Token 的 Key 和 Value，在下一步只为新 Token 计算增量表示，并让新 Token 关注已缓存的历史信息。

为减少自回归推理过程中对历史 Token 的重复计算，本文采用带 KV Cache 的 Transformer Decoder 实现增量解码。该机制缓存历史 Token 的 Key 和 Value，并在后续解码步骤中复用，从而降低重复计算开销。与候选重评分阶段共享Encoder Key/Value不同，KV Cache主要用于Transformer Decoder自注意力层，用于缓存历史生成的Token的Key和Value。与候选重评分阶段共享Encoder Key/Value 不同，KV Cache 主要用于Transformer Decoder自注意力层，用于缓存历史生成Token的Key和Value。

最终三种解码方式可以如下表格表示：

|解码方式       | 候选生成  | Transformer | KV Cache | 特点  |
|----------------------|-------------|---------------|-------------|---------|
CTC Greedy      | CTC    | No      | No    | 最低延迟|
CTC Beam + Rescore | CTC Beam | Yes     | No    | 精度/速度平衡|
AR Decoder      |Transformer | Yes     | Yes    | 最高建模能力|


# Experiments
## Experimental Setup
### Datasets
本文针对移动端离线中文短语音识别场景进行实验评估。训练数据由多个公开中文语音数据集以及内部场景数据组成、覆盖日常语音、多人会议、网络语音以及垂直领域指令等多种语音环境。
- Fleurs-test
- WenetSpeech TEST_MEETING
- WenetSpeech TEST_NET
- AISHELL-3
- CommonVoice
- MagicData
- SpeechIO

这些数据主要用于评估模型在自然语音环境下的识别能力，其中 WenetSpeech TEST_MEETING 和 TEST_NET 包含一定比例背景噪声，用于测试复杂环境下模型鲁棒性。
行业领域测试集包括：
- 地址识别
- 意图短语
- 身份证号码
- 车牌号码
- 警言警语
- 电话号码

这些数据对应移动端语音助手中的典型短语识别任务，具有较短语音长度和较强领域约束。

### Input Feature Extraction
所有输入音频首先转换为16kHz采样率，然后采用短时傅里叶变换（STFT）提取频谱特征。STFT窗口长度设置为25 ms，帧移设置为10 ms，并进一步通过Mel滤波器组转换为80维Log-Mel谱特征。

因此，对于长度为T的音频，输入特征表示为：
$$
  X \in \mathbb{R}^{T \times 80}
$$
为了提升模型对于时间和频率扰动的鲁棒性，训练阶段采用SpecAugment数据增强策略，包括时间掩蔽和频率掩蔽。

### Tokenization
本文采用字符级建模方式，将中文文本转换为字符Token序列。词表大小为7531，并额外增加PAD、UNK、BOS、EOS四个特殊Token。最终词表大小为7535。

### Model Configuration
本文采用第三章介绍的 Citrinet-Transformer Hybrid 模型。

主要配置为：
|Component             | Configuration       |
|-----------------------------------|-----------------------------|
| Encoder             | Citrinet         |
| Feature Dim          | 80             |
| Encoder hidden dim      | 640            |
| Decoder dim          | 512            |
| Decoder layers         | 6             |
| Attention heads        | 16             |
| Vocabulary           | 7535            |
| Max output length       | 128            |

### Training Configuration
模型采用联合CTC/Seq2Seq目标进行训练：
$$
  L = 0.4 L_{CTC} + 0.6 L_{Seq}
$$
训练过程中采用Teacher Forcing和Label Smoothing。为了提高训练效率，采用 mixed precision training。

### Evaluation Metrics
本文采用字符错误率（Character Error Rate, CER）作为主要评价指标：
$$
  CER = \frac{S + I + D}{N_{ref}} \times 100 \%
$$
其中S、D、I分别为替换、删除、插入错误数，$N_{ref}$是参考文本的字符数。由于我们的模型暂时只支持中文，我们使用过滤方法，删除测试集中的英文，去掉标点符号，将阿拉伯数字转为为中文数字。
此外，为评估移动端部署性能，纪录单条音频平均推理时间：
$$
  Latency(ms)
$$

### Decoding Modes
为了分析不同解码策略在识别精度和计算开销之间的差异，本文在同一模型参数下测试三种推理模式。
| Mode        | Decoder              | Description  |
|----------------------|----------------------------------------------|--------|
| Mode0       | CTC Greedy                 |最低延迟|
| Mode1       | CTC prefix Beam + Transformer Rescore |精度速度平衡|
| Mode2       | AR Transformer + KV Cache        | 最高精度 |

## Recognition Accuracy Comparision
<!-- 今天的天气如何 -->
